# Lab 7 — File Handling: Text, JSON & CSV
**Difficulty: Intermediate | ~25 min | Requires Lab 4**

In [1]:
# This lab uses only Python's standard library, but this cell keeps the
# notebook runnable from a fresh kernel. It installs nothing extra.
!pip install --upgrade pip

## Step 1 — Write a Text File with 'w', Read Back with 'r'

In [2]:
# Helper 1: open a file and write/create text into it.
def write_text_file(path, content, mode="w", encoding="utf-8"):
    with open(path, mode, encoding=encoding) as f:  # 'with' auto-closes the file
        f.write(content)


# Helper 2: open a file and read all of it back as one string.
def read_text_file(path, encoding="utf-8"):
    with open(path, "r", encoding=encoding) as f:
        return f.read()


# Write two order lines, then read them back.
write_text_file("orders.txt", "Widget,3,9.99\nGadget,1,14.50\n")
content = read_text_file("orders.txt")

print("--- Text write/read ---")
print(content)

--- Text write/read ---
Widget,3,9.99
Gadget,1,14.50



## Step 2 — Append Mode 'a' + Reading Lines

In [3]:
# mode="a" appends instead of overwriting the existing file.
write_text_file("orders.txt", "Shirt,2,25.00\n", mode="a")

with open("orders.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()  # returns a list, one string per line

print("--- Append + readlines ---")

for i, line in enumerate(lines, 1):  # number from 1 for readability
    print(f"{i}: {line.strip()}")  # strip() removes the trailing newline

--- Append + readlines ---
1: Widget,3,9.99
2: Gadget,1,14.50
3: Shirt,2,25.00


## Step 3 — JSON Dump to File + json.load Read Back

In [4]:
# json module turns Python objects into text (and back again).
import json


def save_json_data(path, data, encoding="utf-8"):
    with open(path, "w", encoding=encoding) as f:
        json.dump(data, f, indent=2)  # indent=2 keeps the file human-readable


def load_json_data(path, encoding="utf-8"):
    with open(path, "r", encoding=encoding) as f:
        return json.load(f)


# The list of orders we will serialize to JSON and CSV.
orders = [
    {"product": "Widget", "qty": 3, "price": 9.99},
    {"product": "Gadget", "qty": 1, "price": 14.50},
    {"product": "Shirt",  "qty": 2, "price": 25.00},
]

# Write orders to a JSON file, then read them back.
save_json_data("orders.json", orders)
loaded = load_json_data("orders.json")  # comes back as a list of dicts

print("--- JSON dump/load ---")
print("Type:", type(loaded).__name__)

for o in loaded:
    print(" ", o)

--- JSON dump/load ---
Type: list
  {'product': 'Widget', 'qty': 3, 'price': 9.99}
  {'product': 'Gadget', 'qty': 1, 'price': 14.5}
  {'product': 'Shirt', 'qty': 2, 'price': 25.0}


## Step 4 — Handle FileNotFoundError with the Context Manager

In [5]:
print("--- Missing file handling ---")

# Attempting to read a file that does not exist raises FileNotFoundError.
try:
    data = read_text_file("no_such_file.txt")
except FileNotFoundError as e:  # catch the specific error, keep running
    print(f"Gracefully caught: {e}")


# Demonstrating the with-open context manager closes the handle
with open("orders.txt", "r", encoding="utf-8") as f:
    first_line = f.readline()  # read just the next line

print("Read first line:", first_line.strip())
print("Handle closed after with block:", f.closed)  # True: context manager did its job

--- Missing file handling ---
Gracefully caught: [Errno 2] No such file or directory: 'no_such_file.txt'
Read first line: Widget,3,9.99
Handle closed after with block: True


## Step 5 — CSV with csv.reader (Raw Rows)

In [6]:
# Rebuild orders.txt as a CSV file with a header row.
import csv

with open("orders.txt", "w", encoding="utf-8") as f:
    f.write("product,qty,price\nWidget,3,9.99\nGadget,1,14.50\nShirt,2,25.00\n")


# csv.reader splits each line into a list of cell values.
print("--- csv.reader raw rows ---")

with open("orders.txt", "r", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(row)

--- csv.reader raw rows ---
['product', 'qty', 'price']
['Widget', '3', '9.99']
['Gadget', '1', '14.50']
['Shirt', '2', '25.00']


## Step 6 — csv.DictReader (Read as Dictionaries)

In [7]:
# csv.DictReader uses the header row as dictionary keys.
print("--- csv.DictReader ---")

with open("orders.txt", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Fieldnames:", reader.fieldnames)  # keys come from the header

    for row in reader:
        print(row)  # OrderedDict like {'product': 'Widget', ...}

--- csv.DictReader ---
Fieldnames: ['product', 'qty', 'price']
{'product': 'Widget', 'qty': '3', 'price': '9.99'}
{'product': 'Gadget', 'qty': '1', 'price': '14.50'}
{'product': 'Shirt', 'qty': '2', 'price': '25.00'}


## Step 7 — csv.DictWriter (Write Header + Rows)

In [8]:
# csv.DictWriter writes dicts using a fixed column order.
fieldnames = ["product", "qty", "price"]

with open("orders.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()  # emit the column names as the first row
    writer.writerows(orders)  # one row per dict in the list


print("--- csv.DictWriter wrote orders.csv ---")

with open("orders.csv", "r", encoding="utf-8") as f:
    print(f.read())

--- csv.DictWriter wrote orders.csv ---
product,qty,price
Widget,3,9.99
Gadget,1,14.5
Shirt,2,25.0



## Step 8 — Inventory Export Pipeline (Text, JSON & CSV)

In [9]:
# --- Write: export the same orders to all three formats ---

# Plain-text: one comma-separated line per order.
with open("inventory.txt", "w", encoding="utf-8") as f:
    for o in orders:
        f.write(f"{o['product']},{o['qty']},{o['price']:.2f}\n")

# JSON: reuse the helper defined earlier.
save_json_data("inventory.json", orders)

# CSV: header + rows via DictWriter.
with open("inventory.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(orders)


# --- Reload: read the data back from all three formats ---

# TXT: split lines into dicts by mapping columns with zip().
txt_orders = [dict(zip(fieldnames, line.strip().split(",")))
              for line in read_text_file("inventory.txt").splitlines()]

# JSON: preserves types automatically.
json_orders = load_json_data("inventory.json")

# CSV: DictReader maps the header row onto each data row.
with open("inventory.csv", "r", encoding="utf-8") as f:
    csv_orders = list(csv.DictReader(f))


# --- Summarize: totals across the reloaded data ---

total_qty = sum(int(r["qty"]) for r in txt_orders)
revenue = sum(float(r["price"]) * int(r["qty"]) for r in json_orders)

print("--- Inventory Pipeline Summary ---")
print(f"Records: {len(txt_orders)} (txt), {len(json_orders)} (json), {len(csv_orders)} (csv)")
print(f"Total items sold: {total_qty}")
print(f"Total revenue: ${revenue:.2f}")

--- Inventory Pipeline Summary ---
Records: 3 (txt), 3 (json), 3 (csv)
Total items sold: 6
Total revenue: $94.47


## Step 9 — Delete Files and Folders Safely

In [10]:
from pathlib import Path
import shutil

tmp_file = Path("delete_me.txt")
tmp_dir = Path("delete_me_folder")


# Create throwaway file and folder to delete.
tmp_file.write_text("temp data", encoding="utf-8")
tmp_dir.mkdir(exist_ok=True)
(tmp_dir / "inside.txt").write_text("nested", encoding="utf-8")


# Delete the file, then the folder (only if they exist).
if tmp_file.exists():
    tmp_file.unlink()  # remove a single file

if tmp_dir.exists():
    shutil.rmtree(tmp_dir)  # remove a folder and everything inside


print("--- Delete files/folders ---")
print("File exists after delete:", tmp_file.exists())
print("Folder exists after delete:", tmp_dir.exists())

--- Delete files/folders ---
File exists after delete: False
Folder exists after delete: False


## Optional Exercise

Extend the pipeline to add a **fourth order** — a "Mug" with qty `4` and price `7.25` — before the export step. Rerun the pipeline and confirm the summary updates: it should report 4 records per format, a total of **10** items sold, and revenue of **$123.47**. Then, in the JSON cell, change the `json.dump` call to use `indent=4` instead of `indent=2` and confirm the file is written with 4-space indentation.